# 14. SQL JSON & Semi-Structured Data: Beginner Guide

### 📝 Universal SQL Execution Order (All Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline ─────────────────────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. SELECT & CASE   ➔ 8. DISTINCT (Dedup)   ➔ 9. ORDER BY (Sorting)         │
│ ➔ 10. LIMIT / OFFSET (Final Page Slice)                                      │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **14. SQL JSON & Semi-Structured Data**. Modern data pipelines frequently ingest semi-structured JSON payloads alongside structured relational tables. This notebook covers JSON property extraction (`json_extract()`, `->`, `->>`), dynamic JSON object generation, relational aggregation to JSON arrays (`json_group_array()`), and parsing nested metadata payloads directly inside relational data engines.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Document Key Navigation: `json_extract(col, '$.path')`
- [x] 🔹 Relational Aggregation to JSON: `json_group_array()`
- [x] 🔹 Type Coercion of Extracted Document Properties
- [x] 🔍 Scenario: Querying Dynamic Merchant Audit & Risk Metadata Payloads










In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Document Key Navigation: `json_extract()`
- **What it does:** Parses a JSON text document and extracts a scalar value matching a JSONPath expression (`$.key_name`).
- **Syntax:** `SELECT json_extract(json_column, '$.path.to.attribute') FROM table_name`
- **Dataset Application & Code Demonstration:** Extracts operating metadata from a dynamic JSON log table.


In [2]:
%%sql
CREATE TABLE IF NOT EXISTS audit_event_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    event_type TEXT,
    payload_json TEXT
);
INSERT OR REPLACE INTO audit_event_logs (log_id, event_type, payload_json) VALUES
    (1, 'PAYMENT_CHECKOUT', '{"ip_address": "192.168.1.10", "risk_score": 0.85, "device": {"os": "iOS", "browser": "Safari"}}'),
    (2, 'PAYMENT_CHECKOUT', '{"ip_address": "10.0.0.5", "risk_score": 0.12, "device": {"os": "Android", "browser": "Chrome"}}');

SELECT 
    log_id,
    event_type,
    json_extract(payload_json, '$.ip_address') AS client_ip,
    CAST(json_extract(payload_json, '$.risk_score') AS REAL) AS risk_score,
    json_extract(payload_json, '$.device.os') AS device_os
FROM audit_event_logs;


'Query Executed Successfully.'

### 🔹 Relational Aggregation to JSON: `json_group_array()`
- **What it does:** Aggregates a series of relational rows into a single valid JSON array string.
- **Syntax:** `SELECT group_col, json_group_array(val_col) FROM table GROUP BY group_col`
- **Dataset Application & Code Demonstration:** Aggregates transaction IDs into a JSON array per customer.


In [3]:
%%sql
SELECT 
    customer_id,
    COUNT(transaction_id) AS tx_count,
    json_group_array(transaction_id) AS transaction_ids_json
FROM (
    SELECT customer_id, transaction_id 
    FROM transactions 
    LIMIT 10
)
GROUP BY customer_id;


,customer_id,tx_count,transaction_ids_json
0,C42098,1,"[""TX110701""]"
1,C55082,1,"[""TX109326""]"
2,C59823,1,"[""TX104210""]"
3,C65296,1,"[""TX103301""]"
4,C68878,1,"[""TX114893""]"
5,C76616,1,"[""TX106376""]"
6,C77715,1,"[""TX106427""]"
7,C81643,1,"[""TX104105""]"
8,C97782,1,"[""TX103284""]"
9,C98117,1,"[""TX114584""]"


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Filtering Relational Records via Nested JSON Properties
- **Objective:** Filter and alert on high-risk checkout events where the extracted JSON risk score exceeds 0.50.
- **Approach:** Apply `json_extract()` inside the `WHERE` predicate filter.


In [4]:
%%sql
SELECT 
    log_id,
    json_extract(payload_json, '$.ip_address') AS ip,
    json_extract(payload_json, '$.risk_score') AS score,
    'FLAGGED_FOR_REVIEW' AS action_status
FROM audit_event_logs
WHERE CAST(json_extract(payload_json, '$.risk_score') AS REAL) > 0.50;


,log_id,ip,score,action_status
0,1,192.168.1.10,0.85,FLAGGED_FOR_REVIEW
